# Uniform sample of everyone (replaces both ancestry rounds)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Sample

In [ ]:
STAGE = "ancestry_round1"
require_ready(cfg, STAGE)
rep, OUT1, OUT2 = Report(cfg, STAGE), single_target(cfg, STAGE), single_target(cfg, "ancestry_round2")
N, SEED = cfg.param("uniform_n"), cfg.param("seed")

aou = pd.read_csv(cfg.input("ancestry_preds"), sep="\t", usecols=["research_id", "ancestry_pred"],
                  dtype={"research_id": str})
pick = np.random.default_rng(SEED).choice(len(aou), size=N, replace=False)
keep = aou.iloc[np.sort(pick)]
rep.note(uniform_n=N, seed=SEED)
rep.count("n_participants", len(aou))
print("sampled", rep.count("n_kept", len(keep)))
rep.check("sample size as asked", len(keep) == N, f"{len(keep)}")
rep.check("no duplicate IDs", not keep["research_id"].duplicated().any())

In [ ]:
comp = pd.DataFrame({"cohort": aou["ancestry_pred"].value_counts(), "sample": keep["ancestry_pred"].value_counts()}).fillna(0).astype(int)
comp["cohort_pct"] = (100 * comp["cohort"] / comp["cohort"].sum()).round(2)
comp["sample_pct"] = (100 * comp["sample"] / comp["sample"].sum()).round(2)
rep.table(comp.reset_index(names="ancestry_pred"), "composition", counts=["cohort", "sample"])

## Write (round 1 and round 2 are the same list: no ancestry gate)

In [ ]:
for stage, uri in [(STAGE, OUT1), ("ancestry_round2", OUT2)]:
    if needs_run(uri, FORCE):
        with record(cfg, stage, uri) as meta:
            get_store(uri).write_text(uri, "\n".join(keep["research_id"]) + "\n")
            meta.update(n_kept=len(keep), uniform_n=N, seed=SEED)
rep.save()